In [ ]:
import os
import io
import re
import fitz
import img2pdf
import numpy as np
import sys
from tqdm.notebook import tqdm
from PIL import Image, ImageDraw, ImageFont
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict
import base64
from pathlib import Path
import json

from openai import OpenAI  # for DeepSeek API compatibility
from dotenv import load_dotenv
load_dotenv()

VLLM_SERVER_URL = os.environ.get("VLLM_SERVER_URL")  
API_KEY = os.environ.get("DEEPSEEK_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME")


# Initialize client (DeepSeek-compatible)
client = OpenAI(
    base_url=VLLM_SERVER_URL,
    api_key=API_KEY,
)

In [75]:
def pdf_to_images_high_quality(pdf_path: str, dpi: int = 144) -> List[Image.Image]:
    """Convert a PDF into high-resolution images."""
    images = []
    pdf = fitz.open(pdf_path)
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)

    for page in pdf:
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        img_data = pix.tobytes("png")
        img = Image.open(io.BytesIO(img_data))
        images.append(img)
    pdf.close()
    return images


def pil_to_pdf_img2pdf(pil_images: List[Image.Image], output_path: str):
    """Convert list of PIL images back to a single PDF."""
    if not pil_images:
        return
    image_bytes = []
    for img in pil_images:
        if img.mode != 'RGB':
            img = img.convert('RGB')
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=95)
        image_bytes.append(buf.getvalue())
    pdf_bytes = img2pdf.convert(image_bytes)
    with open(output_path, "wb") as f:
        f.write(pdf_bytes)


def re_match(text: str):
    """Regex-based extraction of reference bounding boxes."""
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    matches = re.findall(pattern, text, re.DOTALL)
    matches_image, matches_other = [], []
    for m in matches:
        if '<|ref|>image<|/ref|>' in m[0]:
            matches_image.append(m[0])
        else:
            matches_other.append(m[0])
    return matches, matches_image, matches_other


def extract_coordinates_and_label(ref_text, image_width, image_height):
    """Extract bounding box coordinates from OCR reference output."""
    try:
        label_type = ref_text[1]
        coords = eval(ref_text[2])
        return (label_type, coords)
    except Exception as e:
        print("Coordinate extraction error:", e)
        return None


def draw_bounding_boxes(image: Image.Image, refs, jdx: int):
    """Visualize OCR-detected bounding boxes on image."""
    img_copy = image.copy()
    draw = ImageDraw.Draw(img_copy)
    overlay = Image.new('RGBA', img_copy.size, (0, 0, 0, 0))
    draw_overlay = ImageDraw.Draw(overlay)
    font = ImageFont.load_default()

    for ref in refs:
        res = extract_coordinates_and_label(ref, *image.size)
        if not res:
            continue
        label_type, points = res
        color = (np.random.randint(50, 200), np.random.randint(50, 200), np.random.randint(50, 255))
        for box in points:
            x1, y1, x2, y2 = [int(v / 999 * d) for v, d in zip(box, (*image.size, *image.size))]
            draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            draw_overlay.rectangle([x1, y1, x2, y2], fill=color + (25,))
            text_w, text_h = draw.textbbox((0, 0), label_type, font=font)[2:]
            draw.text((x1, max(0, y1 - text_h)), label_type, fill=color, font=font)

    img_copy.paste(overlay, (0, 0), overlay)
    return img_copy

In [76]:
# Configure your test
PDF_FILE = "/Users/jajajou1778/UIT_DOCS_AGENT/firecrawl/data/daa/quydinh_huongdan/huong-dan-chuan-qua-trinh/pdf/547-qd-dhcntt_30-8-2019_qui_dinh_dao_tao_ngoai_ngu_doi_voi_he_chinh_qui_khoa_2019_0_0.pdf"  # Replace with your PDF path
OUTPUT_DIR = "/Users/jajajou1778/UIT_DOCS_AGENT/data/deepseek_ocr_results"

PROMPT = "<image>\n<|grounding|>Convert the document to markdown."
# PROMPT = '<image>\nFree OCR.'
# TODO commonly used prompts
# document: <image>\n<|grounding|>Convert the document to markdown.
# other image: <image>\n<|grounding|>OCR this image.
# without layouts: <image>\nFree OCR.
# figures in document: <image>\nParse the figure.
# general: <image>\nDescribe this image in detail.
# rec: <image>\nLocate <|ref|>xxxx<|/ref|> in the image.
# '先天下之忧而忧'
# .......

In [77]:
class DeepSeekOCR:
    """DeepSeek OCR client for vLLM"""
    
    def __init__(self, client: OpenAI, model_name: str):
        self.client = client
        self.model_name = model_name
    
    def process_image(self, 
                     image: Image.Image, 
                     prompt: str = PROMPT,
                     max_tokens: int = 8000,
                     temperature: float = 0.0) -> str:
        """
        Process a single image with DeepSeek OCR
        
        Args:
            image: PIL Image object
            prompt: OCR instruction prompt
            max_tokens: Maximum tokens in response
            temperature: Sampling temperature
            
        Returns:
            OCR text result
        """
        buf = io.BytesIO()
        image.save(buf, format="PNG")
        img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        
        # Prepare the message
        messages = [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
            ]
        }]
        
        try:
            # Call the model
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
                extra_body={
                    "vllm_xargs": {
                    "ngram_size": 30,
                    "window_size": 90,
                    # "whitelist_token_ids": [128821, 128822],
                    },
                    "skip_special_tokens": False,  # whitelist: <td>, </td>
                }
            )
            
            return response.choices[0].message.content
        
        except Exception as e:
            print(f"Error processing image: {e}")
            return ""
    
    def process_pdf(self, 
                   pdf_path: str,
                   prompt: str = "Please perform OCR on this image and extract all text content.",
                   pages: Optional[List[int]] = None) -> Dict[int, str]:
        """
        Process entire PDF or specific pages
        
        Args:
            pdf_path: Path to PDF file
            prompt: OCR instruction
            pages: List of page numbers to process (None for all)
            
        Returns:
            Dictionary mapping page numbers to OCR results
        """
        # Extract images from PDF
        print(f"Processing PDF: {pdf_path}")
        images = pdf_to_images_high_quality(pdf_path)
        
        # Determine which pages to process
        if pages:
            pages_to_process = [p for p in pages if 0 <= p < len(images)]
        else:
            pages_to_process = list(range(len(images)))
        
        results = {}
        
        for page_num in tqdm(pages_to_process, desc="OCR Processing", dynamic_ncols=True, leave=False):
            print(f"Processing page {page_num + 1}...")
            ocr_result = self.process_image(images[page_num], prompt)
            results[page_num] = ocr_result
        
        return results

# Initialize OCR client
ocr_client = DeepSeekOCR(client, MODEL_NAME)
print("DeepSeek OCR client initialized!")

DeepSeek OCR client initialized!


In [78]:
def display_ocr_results(pdf_path: str, results: Dict[int, str], show_images: bool = True):
    """
    Display OCR results with optional image previews
    
    Args:
        pdf_path: Path to the PDF file
        results: OCR results dictionary
        show_images: Whether to show page images
    """
    if show_images:
        images = pdf_to_images_high_quality(pdf_path)
    
    for page_num, text in results.items():
        print(f"\n{'='*60}")
        print(f"📄 PAGE {page_num + 1}")
        print(f"{'='*60}")
        
        if show_images and page_num < len(images):
            # Display image thumbnail
            fig, ax = plt.subplots(1, 1, figsize=(8, 10))
            ax.imshow(images[page_num])
            ax.axis('off')
            ax.set_title(f"Page {page_num + 1}")
            plt.show()
        
        print("\nExtracted Text:")
        print("-" * 40)
        print(text)
        print("-" * 40)

def save_results(results: Dict[int, str], output_path: str):
    """Save OCR results to a text file"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for page_num, text in results.items():
            f.write(f"\n{'='*60}\n")
            f.write(f"PAGE {page_num + 1}\n")
            f.write(f"{'='*60}\n\n")
            f.write(text)
            f.write("\n")
    
    print(f"Results saved to: {output_path}")

In [79]:
def test_deepseek_ocr(pdf_path: str, 
                      output_dir: str = "./ocr_results",
                      pages: Optional[List[int]] = None,
                      custom_prompt: Optional[str] = None,
                      show_images: bool = True):
    """
    Complete test pipeline for DeepSeek OCR on PDF
    
    Args:
        pdf_path: Path to PDF file
        output_dir: Directory to save results
        pages: Specific pages to process (None for all)
        custom_prompt: Custom OCR prompt
        show_images: Whether to display images
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Default prompt
    if custom_prompt is None:
        custom_prompt = """Please perform OCR on this image. 
        Extract all text content, maintaining the original structure and formatting as much as possible.
        Include any headers, footers, tables, or special formatting."""
    
    print(f"Starting OCR test for: {pdf_path}")
    print(f"Output directory: {output_dir}")
    
    try:
        # Process PDF
        results = ocr_client.process_pdf(
            pdf_path=pdf_path,
            prompt=custom_prompt,
            pages=pages
        )
        
        # Display results
        display_ocr_results(pdf_path, results, show_images)
        
        # Save results
        output_file = os.path.join(output_dir, f"{Path(pdf_path).stem}_ocr.md")
        save_results(results, output_file)
        
        # Save as JSON for programmatic access
        json_file = os.path.join(output_dir, f"{Path(pdf_path).stem}_ocr.json")
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"JSON results saved to: {json_file}")
        
        return results
        
    except Exception as e:
        print(f"Error during OCR test: {e}")
        import traceback
        traceback.print_exc()
        return None

In [80]:
# Configure your test

# Option 1: Process entire PDF
# results = test_deepseek_ocr(
#     pdf_path=PDF_FILE,
#     output_dir=OUTPUT_DIR,
#     show_images=True  # Set to False for faster processing without previews
# )

# Option 2: Process specific pages only
# results = test_deepseek_ocr(
#     pdf_path=PDF_FILE,
#     output_dir=OUTPUT_DIR,
#     pages=[0, 1, 2],  # Process first 3 pages only
#     show_images=True
# )

# Option 3: Use custom prompt for specific extraction
results = test_deepseek_ocr(
    pdf_path=PDF_FILE,
    output_dir=OUTPUT_DIR,
    custom_prompt=PROMPT,
    show_images=False
)

Starting OCR test for: /Users/jajajou1778/UIT_DOCS_AGENT/firecrawl/data/daa/quydinh_huongdan/huong-dan-chuan-qua-trinh/pdf/547-qd-dhcntt_30-8-2019_qui_dinh_dao_tao_ngoai_ngu_doi_voi_he_chinh_qui_khoa_2019_0_0.pdf
Output directory: /Users/jajajou1778/UIT_DOCS_AGENT/data/deepseek_ocr_results
Processing PDF: /Users/jajajou1778/UIT_DOCS_AGENT/firecrawl/data/daa/quydinh_huongdan/huong-dan-chuan-qua-trinh/pdf/547-qd-dhcntt_30-8-2019_qui_dinh_dao_tao_ngoai_ngu_doi_voi_he_chinh_qui_khoa_2019_0_0.pdf


OCR Processing:   0%|          | 0/9 [00:00<?, ?it/s]

Processing page 1...


OCR Processing:  11%|█         | 1/9 [00:06<00:49,  6.19s/it]

Processing page 2...


...

OCR Processing:  78%|███████▊  | 7/9 [00:54<00:19,  9.51s/it]

Processing page 8...


OCR Processing:  89%|████████▉ | 8/9 [00:57<00:07,  7.58s/it]

Processing page 9...



📄 PAGE 1

Extracted Text:
----------------------------------------
547 /QĐ-ĐHCNTT 

<|ref|>title<|/ref|><|det|>[[241, 177, 807, 236]]<|/det|>
# QUYẾT ĐỊNH

<|ref|>text<|/ref|><|det|>[[238, 201, 807, 238]]<|/det|>
Ban hành quy định đào tạo ngoại ngữ đối với hệ đại học chính quy của Trường Đại học Công nghệ Thông tin 

<|ref|>sub_title<|/ref|><|det|>[[210, 253, 838, 273]]<|/det|>
## HIỆU TRƯỞNG TRƯỜNG ĐẠI HỌC CÔNG NGHỆ THÔNG TIN 

<|ref|>text<|/ref|><|det|>[[146, 283, 899, 355]]<|/det|>
Căn cứ Quyết định số 134/2006/QĐ-TTg, ngày 08/6/2006 của Thủ tướng Chính phủ về việc thành lập Trường Đại học Công nghệ thông tin (ĐHCNTT) thuộc Đại học Quốc gia Thành phố Hồ Chí Minh (ĐHQG-HCM); 

<|ref|>text<|/ref|><|det|>[[146, 363, 899, 432]]<|/det|>
Căn cứ Quyết định số 867/QĐ-ĐHQG-TCCB, ngày 17/8/2016 của Giám đốc ĐHQG-HCM ban hành Quy chế tổ chức và hoạt động của trường đại học thành viên và khoa trực thuộc ĐHQG-HCM; 

<|ref|>text<|/ref|><|det|>[[146, 442, 899, 487]]<|/det|>
Căn cứ Quyết định số 1